# Tokamak fusion neutron sources

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nukehub-dev/nucleide/blob/main/notebooks/fusion-source.ipynb)

Build a parametric H-mode tokamak plasma (Miller geometry + Fausser profiles), sample birth neutrons with a seeded generator, and emit MCNP SDEF / Serpent source cards. Then set per-species ion temperatures on a single-fuel config: D-T reacts at the mass-weighted pair temperature, D-D at the deuterium temperature.

> On Colab, run `%pip install "nucleide>=0.16.0"` first. Geometry and profiles below are synthetic hand numbers, no machine data.

In [ ]:
import nucleide.plasma_source as ps

spec = {
    "kind": "parametric",
    "reaction": "dt",
    "major_radius": 620.0,
    "minor_radius": 200.0,
    "elongation": 1.85,
    "triangularity": 0.35,
    "shafranov_factor": 15.0,
    "mode": "H",
    "pedestal_radius": 150.0,
    "ion_density_centre": 1.2e20,
    "ion_density_peaking_factor": 1.1,
    "ion_density_pedestal": 4.0e19,
    "ion_density_separatrix": 3.0e19,
    "ion_temperature_centre": 28.0,
    "ion_temperature_peaking_factor": 2.5,
    "ion_temperature_beta": 2.0,
    "ion_temperature_pedestal": 4.0,
    "ion_temperature_separatrix": 0.1,
}

print(f"<sv>_DT(24 keV) = {ps.reactivity('dt', 24.0):.6g} m3/s")
mom = ps.spectrum_moments("dt", 24.0)
print(f"D-T line at 24 keV: mean {mom['mean_mev']:.6f} MeV, sigma {mom['sigma_mev']:.6f} MeV")
assert mom["mean_mev"] == 14.077934372230146

## Sample and emit cards

The sampler is seeded: identical inputs reproduce the identical stream on a given platform.

In [ ]:
parts = ps.particles(spec, 512, seed=17)
e = parts["energy"]
print(f"{len(e)} particles, mean birth energy {sum(e) / len(e):.4f} MeV")
cards = ps.emit_source_cards(spec, bins=15)
print("SDEF head:", cards["sdef"]["card"].splitlines()[0])
print(
    "spectrum:",
    round(cards["spectrum"]["mean_mev"], 4),
    "+/-",
    round(cards["spectrum"]["sigma_mev"], 4),
    "MeV",
)

## Single-fuel plasma with a species pair

Blank out the mixture by omitting the `fuel` dict and set `(T_D, T_T) = (20, 30)` keV: single-fuel D-T reacts at $T_{DT} = 24$ keV. An equal pair reproduces the shared-temperature kernel bit-for-bit.

In [ ]:
flat = dict(
    spec,
    mode="L",
    triangularity=0.0,
    shafranov_factor=0.0,
    ion_density_centre=1e20,
    ion_density_peaking_factor=0.0,
    ion_density_pedestal=1e20,
    ion_density_separatrix=1e20,
    ion_temperature_centre=20.0,
    ion_temperature_peaking_factor=0.0,
    ion_temperature_beta=1.0,
    ion_temperature_pedestal=20.0,
    ion_temperature_separatrix=20.0,
)
pair = dict(flat, species_temperatures={"D": 20.0, "T": 30.0})
a = ps.particles(flat, 512, seed=17)
b = ps.particles(pair, 512, seed=17)
ea = sum(a["energy"]) / len(a["energy"])
eb = sum(b["energy"]) / len(b["energy"])
print(f"shared-T mean {ea:.4f} MeV vs pair mean {eb:.4f} MeV (pair reacts hotter)")
assert eb > ea

same = dict(flat, species_temperatures={"D": 20.0, "T": 20.0})
c = ps.particles(same, 512, seed=17)
assert all(bool((a[k] == c[k]).all()) for k in ("x", "y", "z", "energy", "weight"))
print("equal-T pair reproduces the shared kernel bit-for-bit")